# 🃏 Treinamento e Evolução de Agentes de IA para Poker (TCC)

Este notebook permite executar o pipeline completo de **treinamento evolutivo adaptativo**, **qualificação individual de métricas acadêmicas** e **gravação de dados** para os Agentes de Poker do seu TCC diretamente na nuvem do Google Colab.

### 🎯 Funcionalidades:
- **Diagnóstico Prescritivo com o Mentor (`TrainerAgent`):** Análise tática pré-treino do agente escolhido.
- **Qualificação Individual de Métricas:** Treino focado em *Taxa de Sucesso de Blefe*, *Fold Equity Gerada*, *Rentabilidade (BB/100)*, *Fator de Agressão (AF)*, *Win Rate*, etc.
- **Adaptação Adversarial (Co-evolução):** Enfrenta os campeões históricos arquivados no *Hall of Fame*.
- **Preservação do Agente Original:** Salva o novo agente otimizado com identificador e nome próprios sem sobrescrever o agente base.
- **Geração de Datasets:** Gravação automática de mãos disputadas e rotuladas em formato `.jsonl`.
- **Gráficos Acadêmicos:** Visualizações da evolução do Fitness por geração e comparação *Antes vs Depois* dos hiperparâmetros.

--- 
## 1. Configuração do Ambiente e Clonagem do Repositório
Executa o clone do repositório oficial do projeto para carregar todos os módulos de regras, agentes e treinamento.

In [ ]:
# 1. Clona o repositório oficial do projeto (se já não estiver na pasta)
import os
import sys
from pathlib import Path

repo_name = 'Poker-TCC'
if not Path(repo_name).exists() and not Path('src').exists():
    print('Clonando repositório Poker-TCC...')
    !git clone https://github.com/AlbertoTotiJ/Poker-TCC.git
    %cd Poker-TCC
elif Path(repo_name).exists():
    %cd Poker-TCC

# Garante que a raiz do projeto esteja no sys.path para os imports
root_path = str(Path('.').resolve())
if root_path not in sys.path:
    sys.path.insert(0, root_path)

print('Diretório ativo:', os.getcwd())
print('Ambiente configurado com sucesso!')

--- 
## 2. Diagnóstico Prévio com o Agente Mentor (`TrainerAgent`)
O Mentor analisa as características fundamentais do agente base selecionado e sugere a melhor métrica para evolução no TCC.

In [ ]:
from src.agents.trainer_agent import TrainerAgent

coach = TrainerAgent()

# Escolha o agente para diagnosticar: 'bluffer', 'mathematical' ou 'gh_math'
agente_escolhido = 'bluffer'

diag = coach.diagnose_agent(agente_escolhido)
print(f'🎓 Mentor de IA: Diagnóstico para {diag.target_agent.upper()}')
print('\n✅ Pontos Fortes:')
for s in diag.strengths:
    print(f'  • {s}')
print('\n⚠️ Vulnerabilidades:')
for v in diag.vulnerabilities:
    print(f'  • {v}')
print(f'\n🎯 Foco Recomendado: {diag.recommended_metric}')
print(f'💡 Prescrição do Treinador: {diag.recommended_focus_rationale}')


--- 
## 3. Configuração dos Parâmetros de Treinamento
Defina o número de gerações, população, mãos por duelo e a métrica-alvo de qualificação.

In [ ]:
# =====================================================================
# PARÂMETROS DA SESSÃO DE TREINAMENTO
# =====================================================================
BASE_AGENT_TYPE = 'bluffer'       # 'bluffer', 'mathematical' ou 'gh_math'
NEW_AGENT_NAME = 'Blefador Elite Colab v1'  # Nome do novo agente preservando o original

# Métrica de foco individual:
# Opções: 'bluff_success_rate', 'fold_equity_realized', 'bb_per_100', 
#         'win_rate', 'aggression_factor', 'pfr', 'vpip' ou 'balanced'
TARGET_METRIC = 'bluff_success_rate'

# Hiperparâmetros do Algoritmo Genético
GENERATIONS = 5                   # Quantidade de ciclos evolutivos (recomendado Colab: 4 a 8)
POPULATION_SIZE = 6               # Quantidade de candidatos por geração (recomendado: 4 a 8)
HANDS_PER_EVAL = 25               # Mãos por candidato em cada avaliação (recomendado: 20 a 50)
RECORD_HANDS = True               # Gravar rodadas no dataset de treino (.jsonl)

print(f'Configuração definida:')
print(f'- Agente Base: {BASE_AGENT_TYPE}')
print(f'- Novo Agente: {NEW_AGENT_NAME}')
print(f'- Métrica Alvo: {TARGET_METRIC}')
print(f'- Escala: {GENERATIONS} gerações x {POPULATION_SIZE} população x {HANDS_PER_EVAL} mãos')

--- 
## 4. Execução do Treinamento Adaptativo
O motor genético iniciará a simulação de partidas, avaliando cada candidato, registrando campeões no *Hall of Fame* e adaptando o pool de adversários.

In [ ]:
from src.training.adaptive_trainer import AdaptiveTrainer
import time

def callback_progresso(info):
    print(f"-> Geração {info['generation']}/{info['total_generations']} concluída | "
          f"Melhor Fitness: {info['best_fitness']} pts ({info['best_tier']}) | "
          f"Mãos gravadas: {info['hands_recorded']} | Duração: {info['duration']}s")

print('🚀 Iniciando treinamento adaptativo no Colab...')
t_start = time.time()

trainer = AdaptiveTrainer(
    agent_type=BASE_AGENT_TYPE,
    target_metric=TARGET_METRIC,
    generations=GENERATIONS,
    population_size=POPULATION_SIZE,
    hands_per_eval=HANDS_PER_EVAL,
    record_hands=RECORD_HANDS,
)

result = trainer.train(progress_callback=callback_progresso)
t_total = time.time() - t_start

print(f'\n✅ Treinamento concluído com sucesso em {t_total:.1f} segundos!')
print(f'Melhor Fitness Alcançado: {result.qualification_card.overall_fitness:.1f} / 100')
print(f'Tier Acadêmico Conquistado: {result.qualification_card.overall_tier}')

--- 
## 5. Salvar Novo Agente Preservando o Original
O novo agente com hiperparâmetros otimizados é cadastrado no `AgentRegistry` sob um novo `agent_id`, garantindo que os agentes base permaneçam intactos.

In [ ]:
from src.training.agent_registry import global_registry

# Salva o novo agente treinado sem alterar os agentes originais
new_agent_meta = global_registry.save_custom_agent(
    name=NEW_AGENT_NAME,
    base_type=BASE_AGENT_TYPE,
    parameters=result.best_parameters,
    qualification=result.qualification_card.to_dict(),
    session_id=result.session_id,
    description=f'Treinado no Colab com foco em {TARGET_METRIC}',
)

print('🎉 Novo Agente Registrado com Sucesso!')
print(f'- ID Único: {new_agent_meta.agent_id}')
print(f'- Nome: {new_agent_meta.name}')
print(f'- Base Original: {new_agent_meta.base_type} (Preservada!)')
print(f'- Arquivo do Modelo: {result.model_filepath}')


--- 
## 6. Boletim de Avaliação & Visualizações Acadêmicas
Exibe a tabela comparativa de hiperparâmetros (*Antes vs Depois*) e gera os gráficos de evolução do Fitness por geração.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Boletim Completo de Notas e Tiers por Métrica
metric_rows = []
for k, item in result.qualification_card.individual_scores.items():
    metric_rows.append({
        'Métrica': k,
        'Valor Medido': round(item.raw_value, 2),
        'Nota (0-100)': round(item.score, 1),
        'Tier Conquistado': item.tier,
        'Descrição': item.description
    })

df_metrics = pd.DataFrame(metric_rows)
print(f"\n📊 BOLETIM DE MÉTRICAS (Fitness Geral: {result.qualification_card.overall_fitness:.1f} pts - {result.qualification_card.overall_tier})")
display(df_metrics)

# 2. Tabela Comparativa de Hiperparâmetros (Antes vs Depois)
param_rows = []
for k, v_orig in result.baseline_parameters.items():
    v_novo = result.best_parameters.get(k, v_orig)
    diff = (v_novo - v_orig) if isinstance(v_novo, (int, float)) and isinstance(v_orig, (int, float)) else 0
    param_rows.append({
        'Parâmetro': k,
        'Original': round(v_orig, 3) if isinstance(v_orig, float) else v_orig,
        'Novo Treinado': round(v_novo, 3) if isinstance(v_novo, float) else v_novo,
        'Variação': f'{diff:+.3f}' if isinstance(diff, float) else diff
    })

df_params = pd.DataFrame(param_rows)
print('\n⚙️ COMPARATIVO DE HIPERPARÂMETROS')
display(df_params)

# 3. Gráfico de Evolução do Fitness por Geração
gens = [s.generation for s in result.generation_history]
best_fits = [s.best_fitness for s in result.generation_history]
avg_fits = [s.average_fitness for s in result.generation_history]

plt.figure(figsize=(9, 4.5))
plt.plot(gens, best_fits, marker='o', linewidth=2.5, color='#1976d2', label='Melhor Fitness da Geração')
plt.plot(gens, avg_fits, marker='s', linewidth=1.8, linestyle='--', color='#ff9800', label='Média da População')
plt.title(f'Curva de Aprendizado Evolutivo - {BASE_AGENT_TYPE.upper()} (Foco: {TARGET_METRIC})', fontsize=12, fontweight='bold')
plt.xlabel('Geração Evolutiva', fontsize=10)
plt.ylabel('Fitness Score (0 - 100)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


--- 
## 7. Download dos Artefatos Gerados para o seu Computador
Baixe o arquivo JSON do novo agente treinado e o dataset de partidas gravadas para importar no site ou utilizar na escrita da sua monografia.

In [ ]:
from google.colab import files

print('📥 Arquivos disponíveis para download:')
print(f'1. Modelo JSON do Agente: {result.model_filepath}')
print(f"2. Dataset JSONL de Partidas: {result.dataset_summary.get('session_file')}")

# Descomente abaixo para baixar diretamente pelo navegador do Colab:
# files.download(result.model_filepath)
# if result.dataset_summary.get('session_file') and Path(result.dataset_summary['session_file']).exists():
#     files.download(result.dataset_summary['session_file'])

--- 
## 8. Como Re-treinar um Agente Salvo (Ajuste Fino / Transfer Learning)
Você pode pegar qualquer agente treinado anteriormente (como o Agent 2) e rodar um novo ciclo de treinamento a partir dele, mudando o foco para outra métrica (por exemplo, adicionando Agressividade sem perder a lucratividade).

In [ ]:
# =====================================================================
# RE-TREINAMENTO (AJUSTE FINO) A PARTIR DE UM AGENTE SALVO
# =====================================================================
# 1. Puxa as novidades do repositório e recarrega na memória
!git pull
import importlib
import src.training.adaptive_trainer
importlib.reload(src.training.adaptive_trainer)
from src.training.adaptive_trainer import AdaptiveTrainer
from src.training.agent_registry import global_registry

# 2. Usa os melhores parâmetros do agente anterior direto da memória:
PARAMETROS_INICIAIS = result.best_parameters
NOVA_METRICA_ALVO = 'aggression_factor'  # Foco em aumentar o Fator de Agressão e PFR
NOME_NOVA_VERSAO = f'{NEW_AGENT_NAME} Agressivo v2'

print(f'🔄 Re-treinando a partir de: {NEW_AGENT_NAME}')
print(f'🎯 Novo foco de qualificação: {NOVA_METRICA_ALVO}')

# 3. Instancia o treinador passando 'initial_parameters'
trainer_v2 = AdaptiveTrainer(
    agent_type=BASE_AGENT_TYPE,
    initial_parameters=PARAMETROS_INICIAIS,  # Inicia com o genoma do agente salvo!
    target_metric=NOVA_METRICA_ALVO,
    generations=4,                           # Ciclo focado em ajuste fino
    population_size=6,
    hands_per_eval=30,
    record_hands=True,
)

result_v2 = trainer_v2.train()

# 4. Salva a nova versão evoluída sem perder o agente original nem a versão anterior
agent_v2_meta = global_registry.save_custom_agent(
    name=NOME_NOVA_VERSAO,
    base_type=BASE_AGENT_TYPE,
    parameters=result_v2.best_parameters,
    qualification=result_v2.qualification_card.to_dict(),
    session_id=result_v2.session_id,
    description=f'Evoluído a partir de {NEW_AGENT_NAME} com foco em {NOVA_METRICA_ALVO}',
)

print('\n🎉 Versão Evoluída Salva com Sucesso!')
print(f'- ID: {agent_v2_meta.agent_id}')
print(f'- Nome: {agent_v2_meta.name}')
print(f'- Novo Fitness Geral: {result_v2.qualification_card.overall_fitness:.1f} pts ({result_v2.qualification_card.overall_tier})')
